# Notebook 26 — Negativos difíciles Open Images V7 para Modelo D

**TFM — Sistema de Detección de Amenazas Armadas en Vídeo**  
Oliver Legarreta García · Universitat Oberta de Catalunya

---

## Objetivo

Descargar de Open Images V7 imágenes con máscaras de segmentación de los confundidores más problemáticos identificados en Modelo C, para entrenar **Modelo D**.

## Análisis de resultados Modelo C — categorías a atacar

| Categoría GAR | FP% Modelo B | FP% Modelo C | Δ | Causa |
|--------------|-------------|-------------|-----|-------|
| N5 Phone relaxed | 20.0% | **70.0%** | +50pp | Pocas imágenes negativas de teléfono |
| N4 Sneaking | 28.6% | **57.1%** | +28pp | Modelo activa en personas sin objeto |
| N8 Phone rec 1h | 60.0% | **73.3%** | +13pp | Teléfono en mano en postura similar a arma |
| N3 Running | 0.0% | **12.5%** | +12pp | Modelo activa en personas sin objeto |
| N10 Bottle | 22.2% | **33.3%** | +11pp | Botella confundida con arma |

## Clases a descargar

| Clase OI V7 | Ataca categorías GAR | Max imágenes |
|------------|---------------------|-------------|
| `Mobile phone` | N5, N6, N7, N8, N9 | 500 |
| `Person` | N3, N4 | 500 |
| `Bottle` | N10, N11 | 300 |
| `Remote control` | N8, N9 | 200 |

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install fiftyone
print('✅ FiftyOne instalado')

In [ ]:
import os
import shutil
import yaml
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

import fiftyone as fo
import fiftyone.zoo as foz

# ── CONFIG ────────────────────────────────────────────────────────────────────
OUT_DIR = '/content/drive/MyDrive/TFM/datasets/openimages_negatives'

# Cada clase descarga en su propia carpeta en /content/ para evitar sobreescrituras
CLASSES_CONFIG = {
    'Mobile phone':   ('oi_mobile_phone',   500, '/content/oi_mobile_phone'),
    'Person':         ('oi_person',         500, '/content/oi_person'),
    'Bottle':         ('oi_bottle',         300, '/content/oi_bottle'),
    'Remote control': ('oi_remote_control', 200, '/content/oi_remote_control'),
}

IMG_DIR = Path(OUT_DIR) / 'images' / 'train'
LBL_DIR = Path(OUT_DIR) / 'labels' / 'train'
IMG_DIR.mkdir(parents=True, exist_ok=True)
LBL_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Config cargada')
print(f'   Clases: {list(CLASSES_CONFIG.keys())}')
print(f'   Total máximo: {sum(v[1] for v in CLASSES_CONFIG.values())} imágenes')
print(f'   Destino: {OUT_DIR}')

---
## 1. Función de conversión de máscaras a YOLOv8 segmentación

In [ ]:
def sample_to_yolo(sample, class_id=0):
    """
    Convierte un sample de Open Images V7 (FiftyOne) a formato YOLOv8 segmentación.
    - Campo de anotaciones: ground_truth
    - Máscara: numpy bool array en coordenadas de imagen completa
    Devuelve lista de strings 'class x1 y1 x2 y2 ...' (polígono normalizado)
    """
    lines = []

    if not hasattr(sample, 'ground_truth') or sample.ground_truth is None:
        return lines

    img = cv2.imread(sample.filepath)
    if img is None:
        return lines
    h, w = img.shape[:2]

    for det in sample.ground_truth.detections:
        if det.mask is None:
            continue

        # Máscara booleana en coordenadas de imagen completa
        mask_u8 = det.mask.astype(np.uint8) * 255

        # Redimensionar si no coincide exactamente
        if mask_u8.shape[:2] != (h, w):
            mask_u8 = cv2.resize(mask_u8, (w, h), interpolation=cv2.INTER_NEAREST)

        contours, _ = cv2.findContours(
            mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if not contours:
            continue

        # Tomar contorno más grande y simplificar
        contour = max(contours, key=cv2.contourArea)
        epsilon = 0.01 * cv2.arcLength(contour, True)
        approx  = cv2.approxPolyDP(contour, epsilon, True)
        if len(approx) < 3:
            continue

        pts    = approx.reshape(-1, 2)
        coords = ' '.join(f'{p[0]/w:.6f} {p[1]/h:.6f}' for p in pts)
        lines.append(f'{class_id} {coords}')

    return lines


print('✅ Función de conversión definida')

---
## 2. Descargar y convertir cada clase por separado

Cada clase descarga en su propia carpeta de `/content/` y se convierte inmediatamente antes de pasar a la siguiente. Esto evita que las descargas se sobreescriban entre sí.

In [ ]:
total_converted = 0
total_skipped   = 0
total_errors    = 0
summary         = {}

for class_name, (ds_name, max_samples, ds_dir) in CLASSES_CONFIG.items():
    print(f'\n{"="*55}')
    print(f'Clase: {class_name} (max {max_samples})')
    print(f'{"="*55}')

    # Eliminar dataset anterior si existe
    if fo.dataset_exists(ds_name):
        fo.delete_dataset(ds_name)

    # Descargar en carpeta propia
    print(f'  Descargando...')
    try:
        ds = foz.load_zoo_dataset(
            'open-images-v7',
            split='train',
            label_types=['segmentations'],
            classes=[class_name],
            max_samples=max_samples,
            dataset_name=ds_name,
            dataset_dir=ds_dir,
        )
        print(f'  ✅ {len(ds)} imágenes descargadas')
    except Exception as e:
        print(f'  ❌ Error en descarga: {e}')
        continue

    # Convertir inmediatamente
    print(f'  Convirtiendo a YOLOv8 seg...')
    class_converted = 0
    class_skipped   = 0
    class_errors    = 0

    for sample in ds:
        src = Path(sample.filepath)
        if not src.exists():
            class_skipped += 1
            continue

        prefix  = class_name[:3].lower().replace(' ', '_')
        fname   = f'oi_{prefix}_{src.name}'
        dst_img = IMG_DIR / fname
        dst_lbl = LBL_DIR / (Path(fname).stem + '.txt')

        try:
            shutil.copy2(src, dst_img)
            lines = sample_to_yolo(sample, class_id=0)
            # Label vacío = negativo puro (sin objeto de interés visible)
            dst_lbl.write_text('\n'.join(lines))
            class_converted += 1
        except Exception as e:
            class_errors += 1

    summary[class_name] = class_converted
    total_converted += class_converted
    total_skipped   += class_skipped
    total_errors    += class_errors

    print(f'  ✅ {class_converted} convertidas | {class_skipped} skipped | {class_errors} errores')

# Resumen final
imgs     = list(IMG_DIR.glob('*'))
lbls     = list(LBL_DIR.glob('*.txt'))
nonempty = [l for l in lbls if l.stat().st_size > 0]

print(f'\n{"="*55}')
print('RESUMEN FINAL')
print(f'{"="*55}')
for class_name, n in summary.items():
    print(f'  {class_name:<20}: {n} imágenes')
print(f'  {"─"*35}')
print(f'  Total convertidas:        {total_converted}')
print(f'  Total skipped:            {total_skipped}')
print(f'  Imágenes en disco:        {len(imgs)}')
print(f'  Labels con máscaras:      {len(nonempty)}')
print(f'  Labels vacíos (neg puro): {len(lbls) - len(nonempty)}')

---
## 3. Verificación visual

In [ ]:
img_files = sorted(IMG_DIR.glob('*'))[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
axes = axes.flatten()

COLORS = [(255,80,80),(80,255,80),(80,80,255),(255,255,80),(255,80,255),(80,255,255)]

for ax, img_path in zip(axes, img_files):
    lbl_path = LBL_DIR / (img_path.stem + '.txt')
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    overlay = img.copy()

    if lbl_path.exists() and lbl_path.stat().st_size > 0:
        for li, line in enumerate(lbl_path.read_text().strip().split('\n')):
            parts = line.strip().split()
            if len(parts) < 7: continue
            coords = list(map(float, parts[1:]))
            pts = np.array([[coords[i]*w, coords[i+1]*h]
                            for i in range(0, len(coords), 2)], dtype=np.int32)
            color = COLORS[li % len(COLORS)]
            cv2.fillPoly(overlay, [pts], color)
        img = cv2.addWeighted(img, 0.5, overlay, 0.5, 0)

    ax.imshow(img)
    ax.set_title(img_path.name[4:10], fontsize=8)
    ax.axis('off')

plt.suptitle('Muestra negativos difíciles Open Images V7', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/muestra_oi_negativos.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Muestra guardada')

---
## 4. Generar YAML y estadísticas finales

In [ ]:
yaml_content = {
    'path': OUT_DIR,
    'train': 'images/train',
    'val':   'images/train',
    'nc': 1,
    'names': {0: 'no_weapon'},
}
yaml_path = Path(OUT_DIR) / 'oi_negatives.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

imgs     = list(IMG_DIR.glob('*'))
lbls     = list(LBL_DIR.glob('*.txt'))
nonempty = [l for l in lbls if l.stat().st_size > 0]

print('='*55)
print('DATASET OPEN IMAGES V7 — LISTO PARA MODELO D')
print('='*55)
print(f'  Imágenes totales:         {len(imgs)}')
print(f'  Labels con máscaras:      {len(nonempty)}')
print(f'  Labels vacíos (neg puro): {len(lbls) - len(nonempty)}')
print(f'  YAML: {yaml_path}')
print()
print('Dataset Modelo D:')
print(f'  Positivos (armas):        8.246 imágenes Roboflow')
print(f'  Negativos LVIS:           108 imágenes')
print(f'  Negativos Open Images V7: {len(imgs)} imágenes')
print(f'  Total negativos:          {108 + len(imgs)}')
print(f'  Ratio pos/neg:            {8246 / (108 + len(imgs)):.1f}')
print()
print('✅ Próximo paso: Notebook 27 — Entrenamiento Modelo D')